In [22]:
# <eval_forecast_cached.py>
# Uses predictions and targets already cached on disk
"""
python /home/saptarishi.dhanuka_asp25/weather/graphcast_dir/graphcast/local_files/eval_forecast_cached.py \ 
--eval_start "2014-08-01" \ 
--eval_end "2014-09-30" \ 
--eval_dataset_choice "imerg" \ 
--vars_to_eval "total_precipitation_6hr" \ 
--params_path_new1 "/Datastorage/saptarishi.dhanuka_asp25/gc_weights/graphcast_1_13_orig_2014-06-01_2014-07-30_FORECAST28_dynamic_weighing_india_mask_expt5.npz" \ 
--params_path_new2 "/Datastorage/saptarishi.dhanuka_asp25/gc_weights/graphcast_1_13_orig_2014-06-01_2014-07-30_FORECAST28_dynamic_weighing_india_mask_expt3.npz" \
--output_csv_path "./evaluation_results/forecast_mse.csv"
"""

# Parameters
year = 2014
start_month = 8
end_month = 9
eval_start = f"{year}-{start_month:02d}-06"
eval_end = f"{year}-{end_month:02d}-24"
dataset_choice = "imerg"
eval_vars = "total_precipitation_6hr"
apath = "/Datastorage/saptarishi.dhanuka_asp25/era5_data/era5_cache/"
params_path_old = '/Datastorage/saptarishi.dhanuka_asp25/gc_weights/origs/graphcast_1_13.npz'

params_path_new1 = '/Datastorage/saptarishi.dhanuka_asp25/gc_weights/graphcast_1_13_orig_shapefile_2024-06-01_2024-07-30_FORECAST28_new.npz'
params_path_new2 = '/Datastorage/saptarishi.dhanuka_asp25/gc_weights/graphcast_1_13_orig_shapefile_2024-06-01_2024-07-30_FORECAST28.npz'
norms_dir = '/Datastorage/saptarishi.dhanuka_asp25/norms_gc/'
plots_dir = 'plots/evals'
latmin, latmax, lonmin, lonmax = 6, 38, 35, 65
plot_timesteps = 7
output_pred_old_dir = '/Datastorage/saptarishi.dhanuka_asp25/preds_dir/'
output_pred_finetuned_dir = '/Datastorage/saptarishi.dhanuka_asp25/preds_dir/'
region_vise = True
regions = ['Central_Northeast', 'Hilly_Regions', 'Northeast', 'Northwest', 'South_Peninsular', 'West_Central']
world_regions = ['India']

import os
import sys
import logging
import argparse
import dataclasses
import xarray as xr
import numpy as np
import pandas as pd
from datetime import datetime
from tqdm.auto import tqdm
import time
import zarr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import LinearSegmentedColormap

import jax
import optax

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.62'
# os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
# os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))
from graphcast import checkpoint, data_utils, rollout, graphcast, normalization
import setup_jax_functions
from plotting import scale, select, plot_data, save_animation, save_static_plot, compute_difference_with_targets_sims, plot_sample_from_ds
from metrics import compute_rmse, compute_mae, compute_bias, compute_acc
from utils import regrid_hres_fine_to_coarse, generate_sample_era5_dataset, grads_fn, parse_args, process_to_graphcast_format, compute_mse, compute_mse_diffs, mask_dbase_india_buffer, mask_dbase_regions

sys.path.append('/home/saptarishi.dhanuka_asp25/weather/graphcast_dir/gc_dist')
import trainer.dataloader
from dist_utils import construct_era5_imerg, construct_era5_imerg_6hourly
from datetime import datetime

print("Imports done")

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
# args = parse_args()

# real_time = args.real_time
# eval_start = args.eval_start
# eval_end = args.eval_end
# dataset_choice = args.eval_dataset_choice
# eval_vars = args.vars_to_eval
# apath = args.eval_data_path
# params_path_old = args.params_path_old
# params_path_new1 = args.params_path_new1
# params_path_new2 = args.params_path_new2

# # Add new argument to the parser in utils.py
# # For now, let's handle it here if it's not in the original file

output_csv_path = "./forecast_evaluation_mse.csv"



# os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.08'

def compute_precipitation_acc_debugged(
    predictions_finetuned: xr.Dataset,
    targets: xr.Dataset,
    climatology_precip: xr.DataArray,
    year: int = 2014,
    region: str = 'India'
) -> xr.DataArray:
    """
    Computes a latitude-weighted Anomaly Correlation Coefficient (ACC) for precipitation.

    This function is debugged to correctly use the .dt accessor for pandas
    datetime properties. It handles timedelta conversion, aligns dataset
    structures, interpolates climatology, and applies latitude weighting.

    Args:
        predictions_finetuned: An xarray Dataset containing the predicted 'total_precipitation_6hr'.
        targets: An xarray Dataset containing the target 'total_precipitation_6hr'.
        climatology_precip: An xarray DataArray of precipitation climatology.

    Returns:
        An xarray DataArray containing the ACC value for each forecast time step.
    """
    # 1. Select the variable and align data structures
    if isinstance(predictions_finetuned, xr.Dataset):
        pred_pr = predictions_finetuned['total_precipitation_6hr'].squeeze('batch')
        targ_pr = targets['total_precipitation_6hr'].squeeze('batch').transpose('time', 'lat', 'lon')

    

    # 2. Create absolute time coordinates
    start_time = pd.to_datetime(f'{year}-08-01 00:00:00')
    # Convert the 'time' coordinate (timedelta) to a pandas Series for easy addition
    time_deltas = pred_pr['time'].to_pandas()
    valid_times = start_time + time_deltas

    # 3. Correctly derive dayofyear and hour using the .dt accessor
    # This is the fix for the AttributeError.
    dayofyear_vals = xr.DataArray(valid_times.dt.dayofyear, coords={'time': pred_pr.time})
    hour_vals = xr.DataArray(valid_times.dt.hour, coords={'time': pred_pr.time})

    # 4. Prepare climatology: rename and interpolate
    climatology_renamed = climatology_precip.rename({'latitude': 'lat', 'longitude': 'lon'})
    climatology_interp = climatology_renamed.sel(
        dayofyear=dayofyear_vals,
        hour=hour_vals
    ).interp_like(pred_pr)

    # 5. Compute anomalies
    pred_pr = pred_pr.copy(data=np.asarray(pred_pr.data))
    targ_pr = targ_pr.copy(data=np.asarray(targ_pr.data))
    
    pred_anomaly = pred_pr - climatology_interp
    targ_anomaly = targ_pr - climatology_interp

    # 6. Compute latitude weights
    weights = np.cos(np.deg2rad(pred_anomaly['lat']))
    weights.name = "weights"

    # 7. Calculate the weighted Anomaly Correlation Coefficient
    pred_anomaly_mean = pred_anomaly.weighted(weights).mean(dim=['lat', 'lon'])
    targ_anomaly_mean = targ_anomaly.weighted(weights).mean(dim=['lat', 'lon'])

    pred_anomaly_dev = pred_anomaly - pred_anomaly_mean
    targ_anomaly_dev = targ_anomaly - targ_anomaly_mean

    numerator = (weights * pred_anomaly_dev * targ_anomaly_dev).sum(dim=['lat', 'lon'])
    denominator_pred_sq = (weights * pred_anomaly_dev**2).sum(dim=['lat', 'lon'])
    denominator_targ_sq = (weights * targ_anomaly_dev**2).sum(dim=['lat', 'lon'])
    
    acc = numerator / np.sqrt(denominator_pred_sq * denominator_targ_sq)
    # acc.name = "anomaly_correlation_coefficient"

    return acc

all_results = []
initialization_dates = pd.to_datetime(pd.date_range(start=eval_start, end=eval_end, freq='D'))

# Max forecast length is 7 days (28 steps of 6 hours)
MAX_FORECAST_STEPS = 28
target_lead_times_str = f"{(MAX_FORECAST_STEPS) * 6}h" # +1 to be safe with slicing
target_lead_times_slice = slice("6h", target_lead_times_str)
forecast_horizons_def = {1: 4, 3: 12, 7: 28} # In 6-hourly steps

current_date = datetime.now().strftime('%Y-%m-%d%H-%M-%S')

print(f"Starting evaluation for {len(initialization_dates)} initialization dates.")
all_results = []
idx = 0


Imports done
Starting evaluation for 50 initialization dates.


In [23]:
initialization_dates

DatetimeIndex(['2014-08-06', '2014-08-07', '2014-08-08', '2014-08-09',
               '2014-08-10', '2014-08-11', '2014-08-12', '2014-08-13',
               '2014-08-14', '2014-08-15', '2014-08-16', '2014-08-17',
               '2014-08-18', '2014-08-19', '2014-08-20', '2014-08-21',
               '2014-08-22', '2014-08-23', '2014-08-24', '2014-08-25',
               '2014-08-26', '2014-08-27', '2014-08-28', '2014-08-29',
               '2014-08-30', '2014-08-31', '2014-09-01', '2014-09-02',
               '2014-09-03', '2014-09-04', '2014-09-05', '2014-09-06',
               '2014-09-07', '2014-09-08', '2014-09-09', '2014-09-10',
               '2014-09-11', '2014-09-12', '2014-09-13', '2014-09-14',
               '2014-09-15', '2014-09-16', '2014-09-17', '2014-09-18',
               '2014-09-19', '2014-09-20', '2014-09-21', '2014-09-22',
               '2014-09-23', '2014-09-24'],
              dtype='datetime64[ns]', freq='D')

In [ ]:
preds_dir = f"/Datastorage/saptarishi.dhanuka_asp25/rolled_out_preds/"
models_to_eval = ['base', 'fine_val', 'fine']

for init_date in tqdm(initialization_dates, desc="Evaluating Forecasts"):
    targets_filename = os.path.join(preds_dir, f"target_init_{init_date}.nc")
    if not os.path.exists(targets_filename):
        tqdm.write(f"Targets file {targets_filename} not found. Skipping.")
        continue
    ground_truth_var = xr.open_dataset(targets_filename, decode_timedelta=True)[eval_vars]
    for model_name in tqdm(models_to_eval, desc="MSE & Diff Calc", leave=False):
        preds_filename = os.path.join(preds_dir, f"{model_name}_init_{init_date}.nc")
        if not os.path.exists(preds_filename):
            tqdm.write(f"Predictions file {preds_filename} not found. Skipping.")
            continue
        predictions = xr.open_dataset(preds_filename, decode_timedelta=True)
        if eval_vars in predictions:
            pred_var = predictions[eval_vars]
        else:
            pred_var = predictions

        times = pred_var.time.values

        for region in tqdm(world_regions, desc=f"World regions for {model_name}", leave=False):
            for timestep in times:
                # Slice predictions and targets to the current forecast horizon
                pred_sliced = pred_var.sel(time=timestep)
                targ_sliced = ground_truth_var.sel(time=timestep)

                # Compute difference for this region
                diff = pred_sliced - targ_sliced
                # *** CHANGE: apply region mask per region name
                if region in regions:
                    diff_region = mask_dbase_regions(diff, region)
                else:
                    diff_region = mask_dbase_india_buffer(diff)

                # Compute MSE over the region
                mse = float((diff_region**2).mean())
                rmse = float(np.sqrt(mse))

                # acc = compute_precipitation_acc_debugged(predictions.sel(lat=slice(6, 37), lon=slice(65, 95)), eval_targets.sel(lat=slice(6, 37), lon=slice(65, 95)), clim_ppt.sel(latitude=slice(37, 6), longitude=slice(65, 95)))

                # Store MSE result
                result_row = {
                    'init_date': init_date.strftime('%Y-%m-%d %H:%M:%S'),
                    'forecast_horizon': str(timestep),
                    'model': model_name,
                    'region': region,
                    'rmse': rmse,
                }
                all_results.append(result_row)

                csv_filename = f'skill_score_{region}_{current_date}.csv'
                pd.DataFrame([result_row]).to_csv(
                    csv_filename,
                    mode='a',
                    header=not os.path.exists(csv_filename),
                    index=False
                )

                logging.debug(f"[{region}] {init_date}, {model_name}, {timestep} RMSE = {rmse:.6f}")

# 6. Save all results to a CSV file
logging.info("Evaluation loop finished. Saving results to CSV.")
results_df = pd.DataFrame(all_results)

output_dir = os.path.dirname(output_csv_path)
if output_dir:
    os.makedirs(output_dir, exist_ok=True)

results_df.to_csv(output_csv_path, index=False)
print(f"Evaluation complete. Results saved to {csv_filename} and args csv")
print("\n--- Sample of Evaluation Results ---")
print(results_df.head())
print("------------------------------------\n")

# </eval_forecast_cached.py>

Evaluating Forecasts:   0%|          | 0/50 [00:00<?, ?it/s]

MSE & Diff Calc:   0%|          | 0/2 [00:00<?, ?it/s]

World regions for base:   0%|          | 0/1 [00:00<?, ?it/s]

Predictions file /Datastorage/saptarishi.dhanuka_asp25/rolled_out_preds/fine_val_init_2014-08-06 00:00:00.nc not found. Skipping.


MSE & Diff Calc:   0%|          | 0/2 [00:00<?, ?it/s]

...

Predictions file /Datastorage/saptarishi.dhanuka_asp25/rolled_out_preds/fine_val_init_2014-09-22 00:00:00.nc not found. Skipping.


MSE & Diff Calc:   0%|          | 0/2 [00:00<?, ?it/s]

World regions for base:   0%|          | 0/1 [00:00<?, ?it/s]

2025-08-31 13:28:41,960 - INFO - Evaluation loop finished. Saving results to CSV.
2025-08-31 13:28:41,974 - INFO - Evaluation complete. Results saved to skill_score.csv and args csv


Predictions file /Datastorage/saptarishi.dhanuka_asp25/rolled_out_preds/fine_val_init_2014-09-23 00:00:00.nc not found. Skipping.
Targets file /Datastorage/saptarishi.dhanuka_asp25/rolled_out_preds/target_init_2014-09-24 00:00:00.nc not found. Skipping.

--- Sample of Evaluation Results ---
             init_date             forecast_horizon model region      rmse
0  2014-08-06 00:00:00   21600000000000 nanoseconds  base  India  0.000056
1  2014-08-06 00:00:00   43200000000000 nanoseconds  base  India  0.000071
2  2014-08-06 00:00:00   64800000000000 nanoseconds  base  India  0.000060
3  2014-08-06 00:00:00   86400000000000 nanoseconds  base  India  0.000051
4  2014-08-06 00:00:00  108000000000000 nanoseconds  base  India  0.000037
------------------------------------

